# Major CS — V4 com forma individual e dificuldade dos adversários

**Pergunta** — a forma dos cinco jogadores e o nível dos adversários ajudam a prever séries, além do histórico da organização?

Reproduzimos a V3, treinamos a V4 e comparamos as previsões nos mesmos confrontos. O estudo usa o CSV integral, as escalas do Major arquivadas no projeto e o mesmo simulador. A previsão da Liquid é uma consequência dos dados e dos parâmetros aprendidos.

1. Reproduzir a base e o treino da V3.
2. Auditar IDs e estatísticas dos jogadores.
3. Medir desempenho anterior e dificuldade do calendário.
4. Avaliar a V4 e medir a contribuição de cada bloco de entradas.
5. Simular o Major 100.000 vezes por modelo e reproduzir as tabelas e os três gráficos.

Execute em ordem no kernel Python do projeto com as dependências de `requirements.txt`. Use `MAJOR_CS_DADOS` para indicar a pasta de fontes caso o notebook esteja em outra localização.

## 1. Ambiente e leitura

O CSV mistura séries completas com mapas individuais. Carregamos o identificador do confronto, o placar, o formato, o horário e os jogadores. As estatísticas produzidas durante o jogo ficam fora das entradas do modelo. Os CSVs por tier servem para identificar os grupos de avaliação e reproduzir o histórico da referência.

In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path, PureWindowsPath
from tempfile import TemporaryDirectory
from unittest.mock import patch

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

candidatas = [Path.cwd(), Path.home() / "Desktop" / "Major" / "Predict-do-major"]
if os.environ.get("MAJOR_CS_DADOS"):
    candidatas.insert(0, Path(os.environ["MAJOR_CS_DADOS"]).expanduser())

PASTA_DADOS = next(
    (pasta.resolve() for pasta in candidatas if (pasta / "cs2_all_tiers_games.csv").is_file()),
    None,
)
if PASTA_DADOS is None:
    raise FileNotFoundError("Indique a pasta Predict-do-major em MAJOR_CS_DADOS.")

# Os módulos originais são usados como dependência local, sem gerar cache na pasta fonte.
pasta_scripts = str(PASTA_DADOS / "scripts")
if pasta_scripts not in sys.path:
    sys.path.insert(0, pasta_scripts)
with patch.object(sys, "dont_write_bytecode", True):
    import reconstruir_major as reconstrucao
    import simular_singapura as simulador
    from modelo_simetrico import ModeloSimetrico

COLUNAS_JOGADORES = [
    f"team{lado}_player{numero}" for lado in (1, 2) for numero in range(1, 6)
]
COLUNAS_DADOS = [
    "match_id", "game_id", "tournament", "datetime", "is_total",
    "team1", "team2", "bestOf", "score1_match", "score2_match", "team1_win",
] + COLUNAS_JOGADORES

caminho_csv = PASTA_DADOS / "cs2_all_tiers_games.csv"
caminho_tier1 = PASTA_DADOS / "cs2_tier1_games.csv"
jogos = pd.read_csv(caminho_csv, usecols=COLUNAS_DADOS, parse_dates=["datetime"])
jogos_tier1 = pd.read_csv(caminho_tier1, usecols=COLUNAS_DADOS, parse_dates=["datetime"])

print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · scikit-learn {sklearn.__version__}")
display(pd.DataFrame([
    {"base": "Todos os tiers", "linhas": len(jogos),
     "totais de séries": jogos["is_total"].sum(), "mapas": jogos["is_total"].eq(False).sum()},
    {"base": "Tier 1", "linhas": len(jogos_tier1),
     "totais de séries": jogos_tier1["is_total"].sum(), "mapas": jogos_tier1["is_total"].eq(False).sum()},
]))

Python 3.15.2 · pandas 3.1.1 · scikit-learn 1.9.1


,base,linhas,totais de séries,mapas
0,Todos os tiers,24109,11482,12627
1,Tier 1,13044,5194,7850


## 2. Limpeza e alvo

### 2.1 Uma linha por série

`is_total` identifica a linha da série. O `match_id` identifica o confronto; os IDs de equipe do CSV variam de partida para partida, como vimos no notebook original. Mantemos a chave de histórico pelo nome exato da equipe e inspecionamos as repetições de `match_id` antes de acumular os jogos.

In [2]:
totais = jogos.loc[jogos["is_total"].eq(True)].copy()
duplicadas = totais.loc[totais["match_id"].duplicated(keep=False)].copy()
duplicadas["jogadores_preenchidos"] = duplicadas[COLUNAS_JOGADORES].notna().sum(axis=1)
display(duplicadas.sort_values(["match_id", "game_id"])[[
    "match_id", "game_id", "tournament", "team1", "team2", "datetime",
    "score1_match", "score2_match", "team1_win", "jogadores_preenchidos",
]])

,match_id,game_id,tournament,team1,team2,datetime,score1_match,score2_match,team1_win,jogadores_preenchidos
5061,7287899,-153225.0,PGL Astana 2026. Закрытая европейская квалифик...,K27,WW Team,2026-03-31 14:00:00,2.0,0.0,1.0,10
5060,7287899,-153223.0,PGL Astana 2026. Closed European Qualifier,K27,WW Team,2026-03-31 14:00:00,2.0,0.0,0.0,10
5100,7298778,-153207.0,Roman Imperium Cup VII,G2 Esports,BETBOOM TEAM,2026-03-30 20:10:00,1.0,2.0,0.0,10
5099,7298778,-153205.0,Roman Imperium Cup VII,G2 Esports,BetBoom Team,2026-03-30 20:10:00,1.0,2.0,0.0,10
5081,7298808,-153218.0,Roman Imperium Cup VII,Acend,G2 Esports,2026-03-31 00:40:00,0.0,2.0,0.0,10
5080,7298808,-153217.0,Roman Imperium Cup VII,Acend,G2 Esports,2026-03-31 00:40:00,0.0,2.0,0.0,9
4964,7298940,-153267.0,PGL Astana 2026. Закрытая европейская квалифик...,Lavked,100 Thieves,2026-04-02 11:00:00,1.0,2.0,0.0,10
4963,7298940,-153266.0,PGL Astana 2026. Closed European Qualifier,Lavked,100 Thieves,2026-04-02 11:00:00,1.0,2.0,0.0,10


### 2.2 Resolver os quatro duplicados e construir o alvo

- Para `7287899` e `7298940`, mantemos a mesma linha retida pelo notebook original. São repetições da qualificação europeia do PGL Astana.
- Para `7298778`, as escalações coincidem e varia a grafia de BetBoom. Mantemos a grafia `BetBoom Team` nesse registro.
- Para `7298808`, mantemos a linha com a escalação mais completa.

O alvo vale 1 quando `score1_match > score2_match`, e 0 quando o segundo time vence. Retiramos séries sem placar decisivo ou sem identificação completa. Mantemos `bestOf` vazio quando o formato está ausente. A imputação será ajustada só com o treino.

In [3]:
TOTAIS_DESCARTADOS = {
    7287899: -153223,
    7298940: -153266,
    7298778: -153207,
    7298808: -153217,
}
ALVO = "team1_win_corrigido"

def limpar_series(base):
    series = base.loc[base["is_total"].eq(True)].copy()
    duplicatas = series.loc[series["match_id"].duplicated(keep=False)]
    for confronto, par in duplicatas.groupby("match_id"):
        comparaveis = par[["datetime", "bestOf", "score1_match", "score2_match"]].copy()
        comparaveis["team1"] = par["team1"].str.casefold()
        comparaveis["team2"] = par["team2"].str.casefold()
        descarte = par["game_id"].eq(TOTAIS_DESCARTADOS.get(confronto))
        if len(par) != 2 or len(comparaveis.drop_duplicates()) != 1 or descarte.sum() != 1:
            raise ValueError(f"Revise a duplicata da série {confronto}.")
        series = series.drop(index=par.index[descarte])

    assert series["match_id"].is_unique, "Cada série precisa ter um identificador único."
    invalidas = (
        series[["match_id", "datetime", "team1", "team2", "score1_match", "score2_match"]]
        .isna().any(axis=1)
        | series["team1"].eq(series["team2"])
        | series["score1_match"].eq(series["score2_match"])
    )
    rejeitadas = series.loc[invalidas].copy()
    series = series.loc[~invalidas].sort_values(["datetime", "match_id"]).reset_index(drop=True)
    series[ALVO] = series["score1_match"].gt(series["score2_match"]).astype(int)
    return series, rejeitadas

series_v3, rejeitadas = limpar_series(jogos)
series_tier1, rejeitadas_tier1 = limpar_series(jogos_tier1)
assert set(series_tier1["match_id"]).issubset(series_v3["match_id"])
print(f"Séries aproveitadas · todos os tiers {len(series_v3):,} · Tier 1 {len(series_tier1):,}")
print(f"Séries sem formato informado: {series_v3['bestOf'].isna().sum()}")
print(f"Indicadores originais divergentes do placar: {(series_v3['team1_win'].notna() & series_v3['team1_win'].ne(series_v3[ALVO])).sum()}")
display(rejeitadas[["match_id", "team1", "team2", "score1_match", "score2_match"]])

Séries aproveitadas · todos os tiers 11,471 · Tier 1 5,190
Séries sem formato informado: 119
Indicadores originais divergentes do placar: 4891


,match_id,team1,team2,score1_match,score2_match
3117,10095505,США,Канада,NaN,NaN
4771,7299208,FaZe Clan,CYBERSHOKE Esports,1.0,1.0
4830,7298927,NRG Esports,Team Voca,1.0,1.0
4881,7299053,G2 Esports,GamerLegion,1.0,1.0
5067,7298791,Passion UA,FaZe Clan,2.0,2.0
16122,1629909,ALTERNATE aTTaX,THE SUSPECT,1.0,1.0
23528,10064713,NaN,NaN,NaN,NaN


### 2.3 Etiquetar o tier para a avaliação

Cruzamos `match_id` com os três arquivos por tier. A etiqueta se refere à competição registrada no dataset e serve só para diagnóstico. Séries presentes exclusivamente no CSV completo continuam na base com a etiqueta **Sem tier identificado**.

In [4]:
ids_por_tier = {}
for numero in (1, 2, 3):
    dados_tier = pd.read_csv(
        PASTA_DADOS / f"cs2_tier{numero}_games.csv", usecols=["match_id", "is_total"]
    )
    ids_por_tier[f"Tier {numero}"] = set(dados_tier.loc[dados_tier["is_total"], "match_id"])

tier_por_id = {confronto: tier for tier, ids in ids_por_tier.items() for confronto in ids}
assert len(tier_por_id) == sum(map(len, ids_por_tier.values())), "Há IDs em mais de um tier."
series_v3["tier"] = series_v3["match_id"].map(tier_por_id).fillna("Sem tier identificado")
display(series_v3.groupby("tier").agg(
    series=("match_id", "size"), de=("datetime", "min"), ate=("datetime", "max")
))

,series,de,ate
tier,,,
Sem tier identificado,22,2026-11-12 13:00:00,2026-11-14 21:30:00
Tier 1,5190,2023-01-10 09:30:00,2026-11-14 13:30:00
Tier 2,5352,2023-10-05 20:00:00,2026-11-01 22:00:00
Tier 3,907,2023-10-11 16:00:00,2026-11-01 16:00:00


## 3. Histórico disponível antes de cada jogo

### 3.1 Uma perspectiva por equipe

Cada série vira duas participações. Se A venceu, A recebe uma vitória e B recebe uma derrota. Somamos participações e vitórias por nome e horário antes de acumular. Assim, duas partidas com o mesmo horário ficam juntas no histórico e só ficam disponíveis para previsões posteriores.

In [5]:
def acumular_historico(series):
    participacoes = pd.concat([
        series[["datetime", "team1", ALVO]].rename(columns={"team1": "equipe", ALVO: "venceu"}),
        series[["datetime", "team2", ALVO]].rename(columns={"team2": "equipe", ALVO: "venceu"})
        .assign(venceu=lambda tabela: 1 - tabela["venceu"]),
    ], ignore_index=True)
    blocos = (
        participacoes.groupby(["equipe", "datetime"], as_index=False)
        .agg(jogos_no_horario=("venceu", "size"), vitorias_no_horario=("venceu", "sum"))
        .sort_values("datetime")
    )
    por_equipe = blocos.groupby("equipe", sort=False)
    blocos["jogos_anteriores"] = por_equipe["jogos_no_horario"].cumsum()
    blocos["vitorias_anteriores"] = por_equipe["vitorias_no_horario"].cumsum()
    return blocos

historico_v3 = acumular_historico(series_v3)
historico_tier1 = acumular_historico(series_tier1)
assert historico_v3["jogos_no_horario"].sum() == 2 * len(series_v3)
assert historico_v3["vitorias_no_horario"].sum() == len(series_v3)
display(historico_v3.loc[historico_v3["equipe"].eq("100 Thieves")].head(5))

,equipe,datetime,jogos_no_horario,vitorias_no_horario,jogos_anteriores,vitorias_anteriores
19,100 Thieves,2026-02-20 20:10:00,1,0,1,0
20,100 Thieves,2026-02-20 21:30:00,1,1,2,1
21,100 Thieves,2026-02-21 00:30:00,1,1,3,2
22,100 Thieves,2026-02-21 17:30:00,1,0,4,2
23,100 Thieves,2026-03-31 17:00:00,1,0,5,2


### 3.2 Ligar cada confronto ao último bloco estritamente anterior

Para cada lado, buscamos o último acumulado com horário menor que o do jogo. Uma estreia começa com zero jogos, zero vitórias e taxa ausente. A referência consulta só o histórico Tier 1, inclusive quando é usada para prever uma partida de Tier 2 ou 3. A V3 consulta o histórico completo para os mesmos confrontos.

In [6]:
CONTAGENS = ["jogos_anteriores", "vitorias_anteriores"]
ATRIBUTOS_HISTORICO = CONTAGENS + ["taxa_vitorias_anteriores"]

def juntar_historico(series, historico):
    saida = series.copy()
    consulta = historico[["equipe", "datetime"] + CONTAGENS].rename(
        columns={"datetime": "horario_historico"}
    )
    for lado in (1, 2):
        time = f"team{lado}"
        atual = series[["match_id", "datetime", time]].rename(columns={time: "equipe"})
        anterior = pd.merge_asof(
            atual.sort_values("datetime"), consulta,
            left_on="datetime", right_on="horario_historico", by="equipe",
            direction="backward", allow_exact_matches=False,
        )
        tem_historico = anterior["horario_historico"].notna()
        assert anterior.loc[tem_historico, "horario_historico"].lt(
            anterior.loc[tem_historico, "datetime"]
        ).all()
        anterior[CONTAGENS] = anterior[CONTAGENS].fillna(0).astype(int)
        anterior["taxa_vitorias_anteriores"] = (
            anterior["vitorias_anteriores"] / anterior["jogos_anteriores"].replace(0, np.nan)
        )
        colunas = {atributo: f"{time}_{atributo}" for atributo in ATRIBUTOS_HISTORICO}
        saida = saida.merge(
            anterior[["match_id"] + ATRIBUTOS_HISTORICO].rename(columns=colunas),
            on="match_id", validate="one_to_one",
        )
    for atributo in ATRIBUTOS_HISTORICO:
        saida[f"dif_{atributo}"] = saida[f"team1_{atributo}"] - saida[f"team2_{atributo}"]
    return saida

dados_v3 = juntar_historico(series_v3, historico_v3)
dados_ref = juntar_historico(series_v3, historico_tier1)
assert (dados_v3["team1_jogos_anteriores"] >= dados_ref["team1_jogos_anteriores"]).all()
assert (dados_v3["team2_jogos_anteriores"] >= dados_ref["team2_jogos_anteriores"]).all()
display(dados_v3[["datetime", "team1", "team2", *reconstrucao.V1_COLUMNS, ALVO]].tail(5))

,datetime,team1,team2,dif_jogos_anteriores,dif_vitorias_anteriores,dif_taxa_vitorias_anteriores,bestOf,team1_win_corrigido
11466,2026-11-14 16:15:00,Team Buster,Team WorldEdit,27,16,-0.025000,3.0,1
11467,2026-11-14 16:15:00,Team leva2k,Team NS,-1,0,0.500000,3.0,1
11468,2026-11-14 18:00:00,Iberian Soul,FORZE Reload,-1,5,0.087814,3.0,1
11469,2026-11-14 19:00:00,Team PCH3LK1N,Team Jerry,2,-2,-0.317460,3.0,1
11470,2026-11-14 21:30:00,WW Team,Hype Tech,63,33,NaN,3.0,0


## 4. Ranking HLTV e parâmetros da referência

### 4.1 Pontos conhecidos antes da partida

Reusamos a regra do projeto. A edição precisa ter sido publicada antes do dia do confronto, ter no máximo 14 dias e combinar nome com pelo menos três jogadores em comum. A diferença de pontos fica ausente se um dos lados não puder ser confirmado. A validação histórica usa a escalação registrada no CSV. A simulação usa a escalação publicada para o Major. As equivalências intermediárias são produzidas em pasta temporária.

In [7]:
with TemporaryDirectory(prefix="major-v3-ranking-") as pasta_temporaria:
    with (
        patch.object(reconstrucao, "PROJECT", PASTA_DADOS),
        patch.object(reconstrucao, "OUTPUT", Path(pasta_temporaria)),
    ):
        atributos_ranking, auditoria_ranking = reconstrucao.ranking_features(series_v3)

assert len(atributos_ranking) == len(series_v3)
pontuacao = atributos_ranking[["match_id", "dif_pontos_hltv"]]
dados_v3 = dados_v3.merge(pontuacao, on="match_id", validate="one_to_one")
dados_ref = dados_ref.merge(pontuacao, on="match_id", validate="one_to_one")

display(dados_v3.groupby("tier")["dif_pontos_hltv"].agg(
    series="size", ranking_dois_lados="count", cobertura=lambda pontos: pontos.notna().mean()
).round(3))

,series,ranking_dois_lados,cobertura
tier,,,
Sem tier identificado,22,1,0.045
Tier 1,5190,3446,0.664
Tier 2,5352,1609,0.301
Tier 3,907,141,0.155


### 4.2 Conferir a reprodução da versão anterior

O manifesto salvo informa quais atributos foram usados. Conferimos cada um deles, mais o alvo, nas séries Tier 1 do experimento anterior. Essa verificação testa se a V3 poderá ser comparada à referência sem mudar a definição das entradas.

In [8]:
relatorio_ref = json.loads(
    (PASTA_DADOS / "dados_externos" / "reconstrucao" / "ultimo_experimento.json").read_bytes()
)
pasta_experimento = PASTA_DADOS / "experimentos" / PureWindowsPath(
    relatorio_ref["pasta_experimento"]
).name
selecao_ref = json.loads((pasta_experimento / "selecao_antes_do_teste.json").read_bytes())
ENTRADAS = selecao_ref["entradas"]
assert selecao_ref["modelo"] == "V1 + ranking (simetrico)"
assert ENTRADAS == reconstrucao.V1_COLUMNS + ["dif_pontos_hltv"]

colunas_comparadas = ENTRADAS + [ALVO]
base_ref_salva = pd.read_csv(
    pasta_experimento / "base_experimentos.csv", usecols=["match_id", *colunas_comparadas]
).set_index("match_id").sort_index()[colunas_comparadas]
base_ref_recalculada = dados_ref.loc[dados_ref["tier"].eq("Tier 1")].set_index("match_id")
pd.testing.assert_frame_equal(
    base_ref_recalculada.sort_index()[colunas_comparadas], base_ref_salva,
    check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-12,
)
modelo_ref = joblib.load(pasta_experimento / "modelo_selecionado.joblib")
assert list(modelo_ref.modelo.feature_names_in_) == ENTRADAS
print(f"{len(base_ref_salva):,} séries Tier 1 conferidas atributo por atributo.")
print(ENTRADAS)

5,190 séries Tier 1 conferidas atributo por atributo.
['dif_jogos_anteriores', 'dif_vitorias_anteriores', 'dif_taxa_vitorias_anteriores', 'bestOf', 'dif_pontos_hltv']


## 5. Separação cronológica e treino

### 5.1 Manter os cortes do primeiro notebook

| Conjunto | Período | Uso |
|---|---|---|
| Treino | Antes de 01/01/2026 | Ajustar os parâmetros e as transformações |
| Validação | De 01/01/2026 até 30/06/2026 | Comparar o comportamento das versões |
| Teste | A partir de 01/07/2026 | Medir o resultado fora do período de treino |

A avaliação usa o horário registrado no CSV como referência temporal. Uma série do teste pode consultar resultados de séries com horário anterior, inclusive no mesmo período. O modelo e o preenchimento de valores ausentes ficam congelados durante a avaliação.

In [9]:
INICIO_VALIDACAO = pd.Timestamp("2026-01-01")
INICIO_TESTE = pd.Timestamp("2026-07-01")
data = dados_v3["datetime"]
particoes = {
    "Treino": data.lt(INICIO_VALIDACAO),
    "Validação": data.ge(INICIO_VALIDACAO) & data.lt(INICIO_TESTE),
    "Teste": data.ge(INICIO_TESTE),
}
assert dados_v3["match_id"].equals(dados_ref["match_id"])
assert (sum(mascara.astype(int) for mascara in particoes.values()) == 1).all()
display(pd.DataFrame([
    {"conjunto": nome, "total": int(mascara.sum()),
     "Tier 1": int((mascara & dados_v3["tier"].eq("Tier 1")).sum()),
     "Tier 2 e 3": int((mascara & dados_v3["tier"].isin(["Tier 2", "Tier 3"])).sum()),
     "Sem tier identificado": int((mascara & dados_v3["tier"].eq("Sem tier identificado")).sum()),
     "de": data[mascara].min(), "até": data[mascara].max()}
    for nome, mascara in particoes.items()
]).set_index("conjunto"))

,total,Tier 1,Tier 2 e 3,Sem tier identificado,de,até
conjunto,,,,,,
Treino,8109,3775,4334,0,2023-01-10 09:30:00,2025-12-28 18:15:00
Validação,1844,642,1202,0,2026-01-03 08:00:00,2026-06-30 21:20:00
Teste,1518,773,723,22,2026-07-01 11:45:00,2026-11-14 21:30:00


### 5.2 Treinar a V3 com o mesmo pipeline

O pipeline preenche valores ausentes com a mediana e acrescenta indicadores de ausência, padroniza as entradas e treina a regressão logística com `max_iter=1000`. Usamos a mesma função de criação e conferimos os parâmetros contra o modelo salvo. A probabilidade final usa as duas ordens da partida, com `bestOf` fixo e diferenças invertidas.

In [10]:
treino = dados_v3.loc[particoes["Treino"]]
pipeline_v3 = reconstrucao.new_pipeline()
for nome, etapa in pipeline_v3.named_steps.items():
    parametros_ref = modelo_ref.modelo.named_steps[nome].get_params(deep=False)
    assert json.dumps(etapa.get_params(deep=False), sort_keys=True) == json.dumps(parametros_ref, sort_keys=True)

pipeline_v3.fit(treino[ENTRADAS], treino[ALVO])
modelo_v3 = ModeloSimetrico(pipeline_v3)

amostra = treino[ENTRADAS].head(30)
invertida = amostra.copy()
diferenças = [nome for nome in ENTRADAS if nome.startswith("dif_")]
invertida[diferenças] = -invertida[diferenças]
np.testing.assert_allclose(
    modelo_v3.predict_proba(amostra)[:, 1] + modelo_v3.predict_proba(invertida)[:, 1], 1.0,
)
print(f"V3 treinada com {len(treino):,} séries.")
display(pd.DataFrame({
    "atributo": pipeline_v3.named_steps["imputacao"].statistics_,
}, index=ENTRADAS).rename(columns={"atributo": "mediana no treino"}))

V3 treinada com 8,109 séries.


,mediana no treino
dif_jogos_anteriores,3.000000
dif_vitorias_anteriores,2.000000
dif_taxa_vitorias_anteriores,0.015385
bestOf,3.000000
dif_pontos_hltv,11.000000


## 6. Construir o histórico dos jogadores

### 6.1 Identidade e qualidade das estatísticas

Usamos o ID local do jogador nas séries históricas e a linha de total retida na limpeza. A participação mantém o histórico individual quando o jogador troca de equipe. IDs repetidos dentro da mesma série são removidos dessa série por ambiguidade. O saldo por mapa é `(kills − mortes) / mapas jogados`. ADR e KAST vêm das colunas do CSV.

In [11]:
# Uma participação por jogador e série
from collections import defaultdict

campos_individuais = ["id", "adr", "kast", "kills", "deaths"]
prefixos = [f"team{lado}_player{numero}" for lado in (1, 2) for numero in range(1, 6)]
colunas_individuais = [
    f"{prefixo}_{campo}" for prefixo in prefixos for campo in campos_individuais
]
totais_jogadores = pd.read_csv(
    caminho_csv,
    usecols=["match_id", "game_id", "is_total", "games_played", *colunas_individuais],
)
totais_jogadores = totais_jogadores.loc[totais_jogadores["is_total"].eq(True)]
series_com_jogadores = series_v3.merge(
    totais_jogadores.drop(columns="is_total"), on=["match_id", "game_id"], validate="one_to_one",
)
assert series_com_jogadores["match_id"].equals(series_v3["match_id"])

partes = []
for lado in (1, 2):
    for posicao in range(1, 6):
        prefixo = f"team{lado}_player{posicao}"
        colunas = {f"{prefixo}_{campo}": campo for campo in campos_individuais}
        colunas[prefixo] = "nick_csv"
        partes.append(
            series_com_jogadores[["match_id", "datetime", "tier", "games_played", *colunas]]
            .rename(columns=colunas)
            .assign(lado=lado, posicao=posicao)
        )

slots = pd.concat(partes, ignore_index=True)
# Um ID repetido no mesmo confronto é tratado como identidade ambígua nessa série.
ids_validos = slots.loc[slots["id"].notna()].copy()
conflitos = ids_validos.duplicated(["match_id", "id"], keep=False)
jogadores = ids_validos.loc[~conflitos].copy()
jogadores["id"] = jogadores["id"].astype("int64")
jogadores["dia"] = jogadores["datetime"].dt.normalize()
jogadores["saldo_mapa"] = (
    (jogadores["kills"] - jogadores["deaths"]) / jogadores["games_played"].where(lambda s: s.gt(0))
)
assert len(slots) == 10 * len(series_v3)

auditoria_jogadores = (
    slots.assign(id_observado=slots["id"].notna(), adr_observado=slots["adr"].notna())
    .groupby("tier")
    .agg(slots=("match_id", "size"), ids=("id_observado", "sum"), adr=("adr_observado", "sum"))
)
auditoria_jogadores["IDs por série"] = 10 * auditoria_jogadores["ids"] / auditoria_jogadores["slots"]
auditoria_jogadores["Valores de ADR por série"] = 10 * auditoria_jogadores["adr"] / auditoria_jogadores["slots"]
print(f"{jogadores['id'].nunique():,} IDs com histórico; {int(conflitos.sum())} participações ambíguas removidas.")
display(auditoria_jogadores[["IDs por série", "Valores de ADR por série"]].round(2))

1,569 IDs com histórico; 48 participações ambíguas removidas.


,IDs por série,Valores de ADR por série
tier,,
Sem tier identificado,6.95,0.86
Tier 1,9.32,7.03
Tier 2,8.77,3.61
Tier 3,6.97,2.30


### 6.2 Medir a dificuldade de cada calendário

O Elo dá uma referência de força às equipes que os jogadores representaram e enfrentaram. A nota começa em 1500, com ajuste de 24 pontos multiplicado pela diferença entre resultado e expectativa. Todas as partidas de um dia consultam as notas do início do dia.

Para distinguir os níveis de competição, usamos a exposição anterior dos jogadores a eventos de Tier 1, 2 e 3, codificados como 2, 1 e 0, e a proporção de participações no Tier 3. Esses dois valores formam um índice de exposição entre 0 e 1 por escalação. A diferença dos índices entra como um ajuste nas chances de vitória e troca de sinal ao inverter os times. O tier do próprio jogo a prever fica fora das entradas.

In [12]:
# Elo antes do dia do confronto e contexto do tier nas séries históricas
ELO_INICIAL = 1500.0
K_ELO = 24.0
NIVEL_POR_TIER = {"Tier 1": 2.0, "Tier 2": 1.0, "Tier 3": 0.0}


def elo_anterior(series):
    notas = {}
    registros = []
    base = series[["match_id", "datetime", "team1", "team2", ALVO]].sort_values("datetime")
    for _, dia in base.groupby(base["datetime"].dt.normalize(), sort=True):
        incrementos = defaultdict(float)
        colunas = ["match_id", "team1", "team2", ALVO]
        for confronto, equipe_a, equipe_b, venceu_a in dia[colunas].itertuples(index=False, name=None):
            nota_a = notas.get(equipe_a, ELO_INICIAL)
            nota_b = notas.get(equipe_b, ELO_INICIAL)
            esperado_a = 1.0 / (1.0 + 10.0 ** ((nota_b - nota_a) / 400.0))
            ajuste = K_ELO * (venceu_a - esperado_a)
            incrementos[equipe_a] += ajuste
            incrementos[equipe_b] -= ajuste
            registros.extend([
                (confronto, 1, nota_a, nota_b),
                (confronto, 2, nota_b, nota_a),
            ])
        # Atualizar no fim do dia mantém todas as previsões do dia fora dos seus resultados.
        for equipe, ajuste in incrementos.items():
            notas[equipe] = notas.get(equipe, ELO_INICIAL) + ajuste
    return pd.DataFrame(registros, columns=["match_id", "lado", "elo_equipe", "elo_rival"])


contexto = elo_anterior(series_v3)
assert len(contexto) == 2 * len(series_v3)
jogadores = jogadores.merge(contexto, how="left", on=["match_id", "lado"], validate="many_to_one")
jogadores["nivel_eventos"] = jogadores["tier"].map(NIVEL_POR_TIER)
jogadores["fracao_tier3"] = jogadores["nivel_eventos"].eq(0).astype(float).where(
    jogadores["nivel_eventos"].notna()
)
display(jogadores.groupby("tier").agg(
    participacoes=("id", "size"),
    elo_equipe_mediano=("elo_equipe", "median"),
    elo_rival_mediano=("elo_rival", "median"),
).round(1))

,participacoes,elo_equipe_mediano,elo_rival_mediano
tier,,,
Sem tier identificado,153,1500.0,1500.0
Tier 1,48351,1579.8,1578.9
Tier 2,46929,1524.0,1519.9
Tier 3,6323,1517.4,1512.0


### 6.3 Pesar a forma recente de cada jogador

Consultamos apenas dias anteriores ao confronto. A cada 90 dias, o peso de uma observação cai pela metade. Uma média exige o equivalente a duas séries observadas. No nível da escalação, cada jogador tem o mesmo peso e o atributo exige três jogadores com histórico. A média dos dois menores saldos é calculada quando há quatro ou mais jogadores com saldo conhecido.

O histórico contém lacunas, sobretudo nos tiers menores. Elas entram como valores ausentes e passam pela mesma imputação da V3, aprendida no treino. Na validação, a composição do time vem da escalação registrada no CSV.

In [13]:
# Forma recente consultada antes do confronto
MEIA_VIDA_DIAS = 90
MINIMO_SERIES_EQUIVALENTES = 2.0
MINIMO_JOGADORES = 3
ORIGEM_PESOS = jogadores["dia"].min()
METRICAS_INDIVIDUAIS = ["adr", "kast", "saldo_mapa"]
CONTEXTO_INDIVIDUAL = ["elo_equipe", "elo_rival", "nivel_eventos", "fracao_tier3"]
MEDIDAS = METRICAS_INDIVIDUAIS + CONTEXTO_INDIVIDUAL


def criar_estados_diarios(participacoes):
    pesos = np.exp2((participacoes["dia"] - ORIGEM_PESOS).dt.days / MEIA_VIDA_DIAS)
    por_dia = participacoes[["id", "dia"]].copy()
    for coluna in MEDIDAS:
        observado = participacoes[coluna].notna()
        por_dia[f"peso_{coluna}"] = pesos.where(observado, 0.0)
        por_dia[f"soma_{coluna}"] = participacoes[coluna].fillna(0.0) * pesos
    blocos = por_dia.groupby(["id", "dia"], as_index=False).sum().sort_values(["id", "dia"])
    medidas_acumuladas = list(por_dia.columns[2:])
    blocos[medidas_acumuladas] = blocos.groupby("id")[medidas_acumuladas].cumsum()
    return blocos.rename(columns={"dia": "dia_anterior"}).sort_values("dia_anterior")


estados_individuais = criar_estados_diarios(jogadores)


def consultar_jogadores(elencos, estados=estados_individuais):
    """Consulte os IDs conhecidos e preserve as vagas sem identificação."""
    conhecidos = elencos.loc[elencos["id"].notna()].copy()
    conhecidos["id"] = conhecidos["id"].astype("int64")
    anteriores = pd.merge_asof(
        conhecidos.sort_values("dia"), estados,
        left_on="dia", right_on="dia_anterior", by="id",
        direction="backward", allow_exact_matches=False,
    )
    observado = anteriores["dia_anterior"].notna()
    assert anteriores.loc[observado, "dia_anterior"].lt(anteriores.loc[observado, "dia"]).all()
    ajuste = np.exp2(-(anteriores["dia"] - ORIGEM_PESOS).dt.days / MEIA_VIDA_DIAS)
    forma = anteriores[["consulta_id", "dia_anterior"]].copy()
    for medida in MEDIDAS:
        peso = anteriores[f"peso_{medida}"]
        peso_efetivo = peso * ajuste
        forma[f"{medida}_recente"] = anteriores[f"soma_{medida}"] / peso.where(
            peso_efetivo.ge(MINIMO_SERIES_EQUIVALENTES)
        )
        if medida == "adr":
            forma["series_equivalentes_adr"] = peso_efetivo
    return elencos.merge(forma, on="consulta_id", how="left", validate="one_to_one")


def resumir_elenco(perfis, por):
    medidas = [f"{nome}_recente" for nome in MEDIDAS]
    grupos_elenco = perfis.groupby(por, sort=False, dropna=False)
    medias = grupos_elenco[medidas].mean().where(grupos_elenco[medidas].count().ge(MINIMO_JOGADORES))
    medias["jogadores_com_forma"] = grupos_elenco["adr_recente"].count()
    medias["jogadores_com_tier"] = grupos_elenco["nivel_eventos_recente"].count()
    medias["saldo_dois_menores"] = grupos_elenco["saldo_mapa_recente"].agg(
        lambda valores: valores.nsmallest(2).mean() if valores.count() >= 4 else np.nan
    )
    return medias.reset_index()


consulta = jogadores[["match_id", "lado", "id", "dia"]].reset_index(drop=True)
consulta["consulta_id"] = consulta.index
forma_pre_jogo = consultar_jogadores(consulta)
elencos_pre_jogo = resumir_elenco(forma_pre_jogo, ["match_id", "lado"])
display(elencos_pre_jogo[[
    "match_id", "lado", "jogadores_com_forma", "adr_recente", "saldo_mapa_recente",
    "nivel_eventos_recente", "fracao_tier3_recente",
]].tail(10).round(3))

,match_id,lado,jogadores_com_forma,adr_recente,saldo_mapa_recente,nivel_eventos_recente,fracao_tier3_recente
22044,10097991,2,0,NaN,NaN,NaN,NaN
22045,7306089,2,0,NaN,NaN,NaN,NaN
22046,10098752,2,0,NaN,NaN,NaN,NaN
22047,10098754,2,0,NaN,NaN,NaN,NaN
22048,10098748,2,0,NaN,NaN,NaN,NaN
22049,10098749,2,0,NaN,NaN,NaN,NaN
22050,7310618,2,0,NaN,NaN,NaN,NaN
22051,7310616,2,0,NaN,NaN,NaN,NaN
22052,7310611,2,0,NaN,NaN,NaN,NaN
22053,7310605,2,0,NaN,NaN,NaN,NaN


### 6.4 Montar as entradas do confronto

Formamos as diferenças entre as escalações e juntamos as entradas da V3. Assim, o modelo mantém a força histórica da equipe e recebe também a forma dos jogadores que estão escalados e o contexto dos rivais que eles enfrentaram.

In [14]:
# Montar diferenças entre as escalações
ATRIBUTOS_FORMA = [f"{nome}_recente" for nome in METRICAS_INDIVIDUAIS] + ["saldo_dois_menores"]
ATRIBUTOS_DIFICULDADE = [f"{nome}_recente" for nome in CONTEXTO_INDIVIDUAL]
ENTRADAS_FORMA = [f"dif_{nome}" for nome in ATRIBUTOS_FORMA]
ENTRADAS_DIFICULDADE = [f"dif_{nome}" for nome in ATRIBUTOS_DIFICULDADE]
ENTRADAS_V4 = [*ENTRADAS, *ENTRADAS_FORMA, *ENTRADAS_DIFICULDADE]

dados_v4 = dados_v3.copy()
for lado in (1, 2):
    recorte = elencos_pre_jogo.loc[elencos_pre_jogo["lado"].eq(lado)].drop(columns="lado")
    recorte = recorte.rename(columns={coluna: f"team{lado}_{coluna}" for coluna in recorte if coluna != "match_id"})
    dados_v4 = dados_v4.merge(recorte, on="match_id", how="left", validate="one_to_one")
for atributo in ATRIBUTOS_FORMA + ATRIBUTOS_DIFICULDADE:
    dados_v4[f"dif_{atributo}"] = dados_v4[f"team1_{atributo}"] - dados_v4[f"team2_{atributo}"]

assert dados_v4["match_id"].equals(dados_v3["match_id"])
assert dados_v4[ENTRADAS].equals(dados_v3[ENTRADAS])
assert set(dados_v4[ENTRADAS_V4].columns).isdisjoint(["tier", ALVO])
display(dados_v4.groupby("tier").agg(
    series=("match_id", "size"),
    forma_dos_dois_lados=("dif_adr_recente", lambda s: s.notna().mean()),
    nivel_dos_dois_lados=("dif_nivel_eventos_recente", lambda s: s.notna().mean()),
).round(3))

,series,forma_dos_dois_lados,nivel_dos_dois_lados
tier,,,
Sem tier identificado,22,0.227,0.409
Tier 1,5190,0.753,0.840
Tier 2,5352,0.568,0.749
Tier 3,907,0.300,0.458


## 7. Treinar e medir a contribuição das novas entradas

Comparamos a V3, a V3 com desempenho individual, a V3 com dificuldade e a V4 com os dois blocos. Mantemos a regressão logística, a simetrização e os cortes cronológicos. A regressão aprende os pesos de Elo e desempenho individual.

O ajuste entre tiers favorece a escalação mais exposta a calendários fortes. Testamos três intensidades na validação, correspondentes a 52,5%, 55% e 57,5% de vitória para um Tier 1 contra um Tier 3 em um exemplo com os demais atributos iguais. Aplicamos a diferença em log-odds e usamos a menor perda logarítmica da validação para escolher a intensidade. Sem exposição suficiente em um dos lados, o ajuste vale zero. O período de teste é o mesmo já usado no projeto, para permitir comparação histórica.

In [15]:
# Regressão da V3 e ajuste de dificuldade que respeita o sentido da diferença de tiers
from scipy.special import expit, logit

COLUNAS_TIERS = ["dif_nivel_eventos_recente", "dif_fracao_tier3_recente"]
COLUNAS_ELO = [coluna for coluna in ENTRADAS_DIFICULDADE if coluna not in COLUNAS_TIERS]
ENTRADAS_PIPELINE_V4 = [coluna for coluna in ENTRADAS_V4 if coluna not in COLUNAS_TIERS]


class ModeloComDificuldade:
    def __init__(self, modelo, entradas_pipeline, multiplicador):
        self.modelo = modelo
        self.entradas_pipeline = entradas_pipeline
        self.multiplicador = multiplicador
        self.classes_ = modelo.classes_

    def predict_proba(self, x):
        # Tier 1 puro tem índice 1, Tier 2 puro tem 0,75 e Tier 3 puro tem 0.
        diferenca_nivel = (
            0.25 * x["dif_nivel_eventos_recente"] - 0.5 * x["dif_fracao_tier3_recente"]
        ).fillna(0.0)
        base = self.modelo.predict_proba(x[self.entradas_pipeline])[:, 1]
        ajustada = expit(logit(base) + self.multiplicador * diferenca_nivel.to_numpy())
        return np.column_stack([1.0 - ajustada, ajustada])


def treinar_pipeline(colunas):
    treino = dados_v4.loc[particoes["Treino"]]
    return reconstrucao.new_pipeline().fit(treino[colunas], treino[ALVO])


pipeline_v4 = treinar_pipeline(ENTRADAS_PIPELINE_V4)
simetrico_v4 = ModeloSimetrico(pipeline_v4)
val = dados_v4.loc[particoes["Validação"]]
calibracao = []
for vantagem in (0.525, 0.55, 0.575):
    candidato = ModeloComDificuldade(simetrico_v4, ENTRADAS_PIPELINE_V4, float(logit(vantagem)))
    predicao = candidato.predict_proba(val[ENTRADAS_V4])[:, 1]
    calibracao.append({
        "Tier 1 x Tier 3 com resto igual (%)": 100 * vantagem,
        "ajuste máximo em log-odds": float(logit(vantagem)),
        "Log Loss de validação": log_loss(val[ALVO], predicao, labels=[0, 1]),
    })
calibracao = pd.DataFrame(calibracao).sort_values("Log Loss de validação").reset_index(drop=True)
ajuste_tier = float(calibracao.loc[0, "ajuste máximo em log-odds"])
modelo_v4 = ModeloComDificuldade(simetrico_v4, ENTRADAS_PIPELINE_V4, ajuste_tier)
pipeline_dificuldade = ENTRADAS + COLUNAS_ELO
modelo_dificuldade = ModeloComDificuldade(
    ModeloSimetrico(treinar_pipeline(pipeline_dificuldade)), pipeline_dificuldade, ajuste_tier,
)
entradas_so_forma = ENTRADAS + ENTRADAS_FORMA
variantes = {
    "V3": (modelo_v3, ENTRADAS),
    "V3 + forma individual": (ModeloSimetrico(treinar_pipeline(entradas_so_forma)), entradas_so_forma),
    "V3 + dificuldade": (modelo_dificuldade, pipeline_dificuldade + COLUNAS_TIERS),
    "V4": (modelo_v4, ENTRADAS_V4),
}

amostra = dados_v4.loc[particoes["Validação"], ENTRADAS_V4].head(64)
invertida = amostra.copy()
colunas_dif = [coluna for coluna in ENTRADAS_V4 if coluna.startswith("dif_")]
invertida[colunas_dif] = -invertida[colunas_dif]
np.testing.assert_allclose(
    modelo_v4.predict_proba(amostra)[:, 1] + modelo_v4.predict_proba(invertida)[:, 1],
    1.0, atol=1e-12,
)
print(f"Treino de {particoes['Treino'].sum():,} séries · V3 com {len(ENTRADAS)} entradas · V4 com {len(ENTRADAS_V4)}.")
assert np.isfinite(pipeline_v4.named_steps["imputacao"].statistics_).all()
display(calibracao.round(6))
display(pd.DataFrame({
    "entrada do pipeline": ENTRADAS_PIPELINE_V4,
    "coeficiente padronizado": pipeline_v4.named_steps["classificador"].coef_[0][:len(ENTRADAS_PIPELINE_V4)],
}).round(4))

# Isolar o efeito aprendido dos tiers em um confronto ilustrativo, com o restante igual.
exemplo_tiers = pd.DataFrame(0.0, index=["mesmo nível", "Tier 1 contra Tier 3", "Tier 3 contra Tier 1"],
                             columns=ENTRADAS_V4)
exemplo_tiers["bestOf"] = 3
exemplo_tiers["dif_nivel_eventos_recente"] = [0.0, 2.0, -2.0]
exemplo_tiers["dif_fracao_tier3_recente"] = [0.0, -1.0, 1.0]
efeito_tiers = pd.DataFrame({
    "chance de A vencer (%)": 100 * modelo_v4.predict_proba(exemplo_tiers)[:, 1],
}, index=exemplo_tiers.index)
assert efeito_tiers.iloc[1, 0] > 50 > efeito_tiers.iloc[2, 0]
np.testing.assert_allclose(efeito_tiers.iloc[1:, 0].sum(), 100.0)
display(efeito_tiers.round(2))

Treino de 8,109 séries · V3 com 5 entradas · V4 com 13.


,Tier 1 x Tier 3 com resto igual (%),ajuste máximo em log-odds,Log Loss de validação
0,52.5,0.100083,0.653517
1,55.0,0.200671,0.653922
2,57.5,0.302281,0.654365


,entrada do pipeline,coeficiente padronizado
0,dif_jogos_anteriores,0.3795
1,dif_vitorias_anteriores,-0.2448
2,dif_taxa_vitorias_anteriores,0.1781
3,bestOf,-0.0131
4,dif_pontos_hltv,0.1837
5,dif_adr_recente,0.0986
6,dif_kast_recente,0.0127
7,dif_saldo_mapa_recente,0.0915
8,dif_saldo_dois_menores,-0.0234
9,dif_elo_equipe_recente,0.0728


,chance de A vencer (%)
mesmo nível,50.0
Tier 1 contra Tier 3,52.5
Tier 3 contra Tier 1,47.5


### 7.1 Validar em séries futuras e em confrontos com desnível

O recorte de desnível considera uma equipe com metade ou mais da exposição recente em Tier 3 e a outra com maioria em Tier 1 e 2. O cálculo vem das escalações antes de cada série. A perda logarítmica e o Brier avaliam as probabilidades, a acurácia conta os vencedores apontados e a AUC mede a ordenação.

In [16]:
# Comparação em validação e teste, com os mesmos confrontos nas quatro versões
LIMITE_POUCO_HISTORICO = 10
GRUPO_POUCO_HISTORICO = "Com equipe abaixo de 10 séries anteriores no Tier 1"
GRUPO_DESNIVEL = "Escalação mais exposta ao Tier 3 contra escalação de Tier 1/2"
colunas_volume = [f"team{lado}_jogos_anteriores" for lado in (1, 2)]
pouco_historico = dados_ref[colunas_volume].lt(LIMITE_POUCO_HISTORICO)
tiers_jogados = dados_v4[["team1_fracao_tier3_recente", "team2_fracao_tier3_recente"]]
predominio_tier3 = tiers_jogados.ge(0.5)
desnivel = tiers_jogados.notna().all(axis=1) & predominio_tier3.sum(axis=1).eq(1)
grupos = {
    "Todas as séries": pd.Series(True, index=dados_v4.index),
    "Apenas Tier 1": dados_v4["tier"].eq("Tier 1"),
    "Tier 2 e 3": dados_v4["tier"].isin(["Tier 2", "Tier 3"]),
    GRUPO_POUCO_HISTORICO: pouco_historico.any(axis=1),
    GRUPO_DESNIVEL: desnivel,
}


def calcular_metricas(alvo, probabilidade):
    return {
        "Log Loss": log_loss(alvo, probabilidade, labels=[0, 1]),
        "Brier": brier_score_loss(alvo, probabilidade),
        "Acurácia": accuracy_score(alvo, (probabilidade > 0.5).astype(int)),
        "AUC": roc_auc_score(alvo, probabilidade) if alvo.nunique() == 2 else np.nan,
    }


def avaliar(conjunto):
    selecao = particoes[conjunto]
    alvo = dados_v4.loc[selecao, ALVO]
    probabilidades = {
        "Referência HLTV": modelo_ref.predict_proba(dados_ref.loc[selecao, ENTRADAS])[:, 1],
        **{
            nome: estimador.predict_proba(dados_v4.loc[selecao, colunas])[:, 1]
            for nome, (estimador, colunas) in variantes.items()
        },
        "50%": 0.5,
    }
    probs = pd.DataFrame(probabilidades, index=alvo.index)
    resultados = []
    for nome, pertence in grupos.items():
        ids = alvo.index[pertence.loc[alvo.index]]
        if len(ids):
            for versao in probs:
                resultados.append({
                    "grupo": nome, "modelo": versao, "séries": len(ids),
                    **calcular_metricas(alvo.loc[ids], probs.loc[ids, versao]),
                })
    return pd.DataFrame(resultados).set_index(["grupo", "modelo"]), probs


metricas_validacao, prob_validacao = avaliar("Validação")
metricas_teste, prob_teste = avaliar("Teste")
print("Validação · janeiro a junho de 2026")
display(metricas_validacao.round(4))
print("Teste · julho a novembro de 2026")
display(metricas_teste.round(4))

Validação · janeiro a junho de 2026
Teste · julho a novembro de 2026


séries  ...     AUC
grupo                                              modelo                         ...        
Todas as séries                                    Referência HLTV          1844  ...  0.6100
                                                   V3                       1844  ...  0.6327
                                                   V3 + forma individual    1844  ...  0.6402
                                                   V3 + dificuldade         1844  ...  0.6321
                                                   V4                       1844  ...  0.6382
                                                   50%                      1844  ...  0.5000
Apenas Tier 1                                      Referência HLTV           642  ...  0.6433
                                                   V3                        642  ...  0.6379
                                                   V3 + forma individual     642  ...  0.6499
                                                   V3 + dificuldade          642  ...  0.6599
                                                   V4                        642  ...  0.6700
                                                   50%                       642  ...  0.5000
Tier 2 e 3                                         Referência HLTV          1202  ...  0.5926
                                                   V3                       1202  ...  0.6337
                                                   V3 + forma individual    1202  ...  0.6402
                                                   V3 + dificuldade         1202  ...  0.6186
                                                   V4                       1202  ...  0.6242
                                                   50%                      1202  ...  0.5000
Com equipe abaixo de 10 séries anteriores no Ti... Referência HLTV          1121  ...  0.6103
                                                   V3                       1121  ...  0.6315
                                                   V3 + forma individual    1121  ...  0.6337
                                                   V3 + dificuldade         1121  ...  0.6285
                                                   V4                       1121  ...  0.6294
                                                   50%                      1121  ...  0.5000
Escalação mais exposta ao Tier 3 contra escalaç... Referência HLTV            44  ...  0.6253
                                                   V3                         44  ...  0.6358
                                                   V3 + forma individual      44  ...  0.6021
                                                   V3 + dificuldade           44  ...  0.7053
                                                   V4                         44  ...  0.6779
                                                   50%                        44  ...  0.5000

[30 rows x 5 columns]

séries  ...     AUC
grupo                                              modelo                         ...        
Todas as séries                                    Referência HLTV          1518  ...  0.6389
                                                   V3                       1518  ...  0.6460
                                                   V3 + forma individual    1518  ...  0.6481
                                                   V3 + dificuldade         1518  ...  0.6444
                                                   V4                       1518  ...  0.6473
                                                   50%                      1518  ...  0.5000
Apenas Tier 1                                      Referência HLTV           773  ...  0.6350
                                                   V3                        773  ...  0.6520
                                                   V3 + forma individual     773  ...  0.6573
                                                   V3 + dificuldade          773  ...  0.6390
                                                   V4                        773  ...  0.6414
                                                   50%                       773  ...  0.5000
Tier 2 e 3                                         Referência HLTV           723  ...  0.6492
                                                   V3                        723  ...  0.6473
                                                   V3 + forma individual     723  ...  0.6474
                                                   V3 + dificuldade          723  ...  0.6575
                                                   V4                        723  ...  0.6603
                                                   50%                       723  ...  0.5000
Com equipe abaixo de 10 séries anteriores no Ti... Referência HLTV           864  ...  0.6607
                                                   V3                        864  ...  0.6620
                                                   V3 + forma individual     864  ...  0.6659
                                                   V3 + dificuldade          864  ...  0.6636
                                                   V4                        864  ...  0.6674
                                                   50%                       864  ...  0.5000
Escalação mais exposta ao Tier 3 contra escalaç... Referência HLTV            20  ...  0.6615
                                                   V3                         20  ...  0.6354
                                                   V3 + forma individual      20  ...  0.6458
                                                   V3 + dificuldade           20  ...  0.6875
                                                   V4                         20  ...  0.6354
                                                   50%                        20  ...  0.5000

[30 rows x 5 columns]

### 7.2 Dar o mesmo peso aos times com pouca cobertura

Calculamos o Brier por equipe e depois tiramos a média. O desvio em pontos percentuais compara a chance prevista com a taxa de vitórias observada. Isso evita que a quantidade de partidas de uma equipe domine o recorte.

In [17]:
# Dar o mesmo peso a cada equipe com pouco histórico no Tier 1
participacoes_teste = []
for lado in (1, 2):
    coluna_equipe = f"team{lado}"
    coluna_jogos = f"{coluna_equipe}_jogos_anteriores"
    ids = dados_ref.index[particoes["Teste"] & pouco_historico[coluna_jogos]]
    tabela = pd.DataFrame({
        "equipe": dados_v4.loc[ids, coluna_equipe],
        "histórico referência": dados_ref.loc[ids, coluna_jogos],
        "histórico V3": dados_v4.loc[ids, coluna_jogos],
        "jogadores com forma": dados_v4.loc[ids, f"team{lado}_jogadores_com_forma"].fillna(0),
        "venceu": dados_v4.loc[ids, ALVO],
        "prob referência": prob_teste.loc[ids, "Referência HLTV"],
        "prob V3": prob_teste.loc[ids, "V3"],
        "prob V4": prob_teste.loc[ids, "V4"],
    })
    if lado == 2:
        inverter = ["venceu", "prob referência", "prob V3", "prob V4"]
        tabela[inverter] = 1.0 - tabela[inverter]
    participacoes_teste.append(tabela)

participacoes_teste = pd.concat(participacoes_teste, ignore_index=True)
diagnostico_equipes = []
for versao in ("referência", "V3", "V4"):
    erro = participacoes_teste[f"prob {versao}"] - participacoes_teste["venceu"]
    por_equipe = participacoes_teste.assign(erro=erro, quadratico=erro.pow(2)).groupby("equipe")
    historico = "histórico referência" if versao == "referência" else "histórico V3"
    diagnostico_equipes.append({
        "modelo": versao,
        "equipes": por_equipe.ngroups,
        "participações": len(participacoes_teste),
        "cobertura de histórico (%)": 100 * participacoes_teste[historico].gt(0).mean(),
        "cobertura de forma de ≥3 jogadores (%)": (
            100 * participacoes_teste["jogadores com forma"].ge(MINIMO_JOGADORES).mean()
            if versao == "V4" else np.nan
        ),
        "Brier médio por equipe": por_equipe["quadratico"].mean().mean(),
        "desvio médio por equipe (p.p.)": 100 * por_equipe["erro"].mean().mean(),
    })
diagnostico_equipes = pd.DataFrame(diagnostico_equipes).set_index("modelo")
display(diagnostico_equipes.round(4))

,equipes,participações,cobertura de histórico (%),cobertura de forma de ≥3 jogadores (%),Brier médio por equipe,desvio médio por equipe (p.p.)
modelo,,,,,,
referência,309,1335,46.2921,NaN,0.2361,11.0395
V3,309,1335,85.3933,NaN,0.2303,10.1442
V4,309,1335,85.3933,30.2622,0.2312,10.1938


### 7.3 Verificar o pipeline treinado

O pacote da V4 fica disponível como `pacote_v4` neste notebook. Verificamos a serialização em memória e comparamos as previsões depois de recarregar o pipeline.

In [18]:
# Verificar que o modelo pode ser serializado e recarregado no notebook
from io import BytesIO

pacote_v4 = {
    "versao": "V4",
    "pipeline": pipeline_v4,
    "entradas": ENTRADAS_V4,
    "entradas_pipeline": ENTRADAS_PIPELINE_V4,
    "ajuste_de_tier_log_odds": ajuste_tier,
    "simetrizacao": "media_da_previsao_direta_e_complemento_da_inversa",
    "treino_antes_de": str(INICIO_VALIDACAO.date()),
    "teste_a_partir_de": str(INICIO_TESTE.date()),
    "meia_vida_dias": MEIA_VIDA_DIAS,
    "elo_inicial": ELO_INICIAL,
    "k_elo": K_ELO,
    "sklearn": sklearn.__version__,
    "csv_sha256": hashlib.sha256(caminho_csv.read_bytes()).hexdigest(),
}
with BytesIO() as memoria:
    joblib.dump(pacote_v4, memoria, compress=3)
    memoria.seek(0)
    recuperado_v4 = joblib.load(memoria)
verificado_v4 = ModeloComDificuldade(
    ModeloSimetrico(recuperado_v4["pipeline"]),
    recuperado_v4["entradas_pipeline"],
    recuperado_v4["ajuste_de_tier_log_odds"],
)
np.testing.assert_allclose(
    verificado_v4.predict_proba(amostra), modelo_v4.predict_proba(amostra),
    atol=1e-12,
)
print("Pipeline V4 serializado e conferido com as mesmas probabilidades.")

Pipeline V4 serializado e conferido com as mesmas probabilidades.


## 8. Simular o Major de Singapura

### 8.1 Fixar as escalações e a data da previsão

Usamos as fontes arquivadas em 17/11/2026, com ranking de 16/11 e partidas locais até 14/11. As consultas de jogadores usam apenas os dados anteriores à data da previsão. Para ligar os apelidos oficiais aos IDs do CSV, reconhecemos equivalências comprovadas e mantemos ambiguidade como ausência.

A escalação de jogo da **9z Globant** usa **history** na vaga de **meyern**. A 9z anunciou o substituto em 07/11 e a ficha da estreia contra a Lynn Vision inclui history. A V4 consulta o histórico individual de history e usa as fontes arquivadas para o histórico da organização. Nesta execução, history somou 0,262 série equivalente de ADR, abaixo do mínimo de duas. As médias de forma da 9z usam os quatro jogadores com dados suficientes. [Anúncio da 9z](https://x.com/9zTeam/status/2119173105876672735) · [Escalação da estreia](https://www.hltv.org/major/match/2400495/9z-vs-lynn-vision-pgl-major-singapore-2026-stage-1)

In [19]:
pasta_major = PASTA_DADOS / "singapura_2026"
ponteiro = json.loads((pasta_major / "ultima_preparacao.json").read_bytes())
fontes_major = pasta_major / PureWindowsPath(ponteiro["pasta_fontes"]).name
configuracao = json.loads((fontes_major / "config_torneio.json").read_bytes())

with patch.object(simulador, "PROJECT", PASTA_DADOS):
    with patch.object(simulador, "base_v1", return_value=(jogos, series_v3)):
        atributos_major, cobertura_major, _ = simulador.snapshots(configuracao, fontes_major)
    with patch.object(simulador, "base_v1", return_value=(jogos_tier1, series_tier1)):
        _, cobertura_major_ref, _ = simulador.snapshots(configuracao, fontes_major)

cobertura_fase1 = (
    cobertura_major.loc[cobertura_major["etapa_entrada"].eq(1)]
    .merge(
        cobertura_major_ref[["chave", "jogos_historico_local", "nome_local"]],
        on="chave", validate="one_to_one", suffixes=("_v3", "_ref"),
    )
    .sort_values("seed_entrada")
)
cobertura_antes = int(cobertura_fase1["nome_local_ref"].notna().sum())
cobertura_depois = int(cobertura_fase1["nome_local_v3"].notna().sum())
print(f"Histórico corroborado na Etapa 1 · referência {cobertura_antes}/16 · V3 {cobertura_depois}/16")
display(cobertura_fase1[[
    "nome_oficial", "nome_local_v3", "jogos_historico_local_ref",
    "jogos_historico_local_v3", "ranking_pontos",
]])

Histórico corroborado na Etapa 1 · referência 8/16 · V3 10/16


,nome_oficial,nome_local_v3,jogos_historico_local_ref,jogos_historico_local_v3,ranking_pontos
0,Ninjas in Pyjamas,Ninjas in Pyjamas,140,229,93
1,PVISION,PARIVISION,144,269,188
2,HEROIC,Heroic,223,274,76
3,FaZe Clan,FaZe Clan,242,268,90
4,Team Liquid,Team Liquid,203,229,128
5,Eternal Fire,Eternal Fire,113,216,76
6,100 Thieves,100 Thieves,0,47,55
7,9z Globant,9z Team,101,123,89
8,M80,M80,119,151,71
9,DENDELE,NaN,0,0,48


In [20]:
# Resolver as escalações arquivadas usando o histórico anterior à previsão
import re

CORTE_MAJOR = pd.Timestamp(configuracao["data_previsao"])
APELIDOS_MAJOR = {
    "naf-fly": "naf",
    "dev1ce": "device",
    "jottaaa": "jotta",
}


def nick_normalizado(texto):
    if pd.isna(texto):
        return None
    nick = str(texto).split("(", 1)[0].strip()
    nick = re.sub(r",\s*[A-Z]{2}$", "", nick).strip().casefold()
    return APELIDOS_MAJOR.get(nick, nick)


identidades = jogadores.loc[
    jogadores["dia"].lt(CORTE_MAJOR) & jogadores["nick_csv"].notna(),
    ["match_id", "lado", "id", "nick_csv", "dia"],
].copy()
identidades["nick"] = identidades["nick_csv"].map(nick_normalizado)
ids_por_nick = identidades.groupby("nick")["id"].agg(set).to_dict()

lados = pd.concat([
    series_v3[["match_id", f"team{lado}"]].rename(columns={f"team{lado}": "nome_equipe"})
    .assign(lado=lado)
    for lado in (1, 2)
], ignore_index=True)
identidades = identidades.merge(lados, on=["match_id", "lado"], validate="many_to_one")
recentes = identidades.loc[identidades["dia"].ge(CORTE_MAJOR - pd.Timedelta(days=90))]
ids_por_nick_e_time = recentes.groupby(["nick", "nome_equipe"])["id"].agg(set).to_dict()
nomes_locais = cobertura_major.set_index("chave")["nome_local"].to_dict()

# Escalação de estreia da 9z, confirmada antes do corte da previsão.
substituicoes_confirmadas = {"9z_globant": {"meyern": "history"}}
escalas_major = []
for time in configuracao["participantes"]:
    chave = time["chave"]
    assert len(time["jogadores"]) == 5
    for inscrito in time["jogadores"]:
        jogador = substituicoes_confirmadas.get(chave, {}).get(inscrito, inscrito)
        nick = nick_normalizado(jogador)
        ids = ids_por_nick.get(nick, set())
        motivo = "apelido único no histórico"
        if len(ids) > 1:
            ids = ids & ids_por_nick_e_time.get((nick, nomes_locais[chave]), set())
            motivo = "apelido e equipe recente confirmada"
        jogador_id = next(iter(ids)) if len(ids) == 1 else None
        escalas_major.append({
            "chave": chave,
            "nome_oficial": time["nome_oficial"],
            "etapa_entrada": time["etapa_entrada"],
            "jogador": jogador,
            "id": jogador_id,
            "identificação": motivo if jogador_id is not None else "sem ID confirmado",
            "dia": CORTE_MAJOR,
        })
escalas_major = pd.DataFrame(escalas_major)
duplicados = escalas_major["id"].notna() & escalas_major["id"].duplicated(keep=False)
escalas_major.loc[duplicados, "id"] = np.nan
escalas_major.loc[duplicados, "identificação"] = "ID repetido em vagas do Major"
escalas_major["consulta_id"] = escalas_major.index
perfis_major = consultar_jogadores(escalas_major)
elencos_major = resumir_elenco(perfis_major, ["chave"])
atributos_v4 = {
    chave: {**atributos_major[chave], **medidas}
    for chave, medidas in elencos_major.set_index("chave")[
        ATRIBUTOS_FORMA + ATRIBUTOS_DIFICULDADE
    ].to_dict("index").items()
}
assert set(atributos_v4) == set(atributos_major)

cobertura_jogadores_major = (
    perfis_major.groupby(["chave", "nome_oficial", "etapa_entrada"], as_index=False)
    .agg(ids_confirmados=("id", "count"), forma_anterior=("adr_recente", "count"))
    .merge(elencos_major[["chave", "nivel_eventos_recente", "fracao_tier3_recente"]],
           on="chave", validate="one_to_one")
    .sort_values(["etapa_entrada", "nome_oficial"])
)
display(cobertura_jogadores_major.drop(columns="chave").round(3))

,nome_oficial,etapa_entrada,ids_confirmados,forma_anterior,nivel_eventos_recente,fracao_tier3_recente
0,100 Thieves,1,5,5,1.417,0.003
2,9z Globant,1,5,4,1.965,0.000
6,Chinggis Warriors,1,5,2,1.958,0.000
7,DENDELE,1,5,5,1.674,0.115
8,Eternal Fire,1,5,5,1.270,0.053
10,FaZe Clan,1,5,5,1.348,0.126
11,FlyQuest,1,5,4,1.928,0.000
15,GOSH,1,5,5,1.233,0.000
16,HEROIC,1,5,5,1.602,0.001
17,Imperial Esports,1,5,5,1.936,0.000


### 8.2 Conferir os confrontos de abertura

Comparamos a probabilidade de vitória em MD1 por confronto. A tabela da Liquid também detalha a forma dos cinco jogadores, com o número equivalente de séries que sustentam as médias.

In [21]:
# Abrir a forma dos jogadores da Liquid e da 9z e as chances da rodada inicial
liquid_jogadores = perfis_major.loc[perfis_major["chave"].eq("liquid"), [
    "jogador", "id", "identificação", "series_equivalentes_adr",
    "adr_recente", "kast_recente", "saldo_mapa_recente", "elo_rival_recente",
    "nivel_eventos_recente", "fracao_tier3_recente",
]].copy()
display(liquid_jogadores.round(3))

chaves, matrizes_v3, _ = simulador.probability_matrices(
    configuracao, atributos_major, modelo_v3, ENTRADAS,
)
chaves_v4, matrizes, _ = simulador.probability_matrices(
    configuracao, atributos_v4, verificado_v4, ENTRADAS_V4,
)
assert chaves == chaves_v4
indice_equipe = {chave: i for i, chave in enumerate(chaves)}
nome_equipe = cobertura_major.set_index("chave")["nome_oficial"].to_dict()
chaves_etapa1 = configuracao["etapas"]["1"]
abertura = pd.DataFrame([
    {
        "Equipe A": nome_equipe[a], "Equipe B": nome_equipe[b],
        "Vitória de A V3 (%)": 100 * matrizes_v3[1][indice_equipe[a], indice_equipe[b]],
        "Vitória de A V4 (%)": 100 * matrizes[1][indice_equipe[a], indice_equipe[b]],
    }
    for a, b in zip(chaves_etapa1[:8], chaves_etapa1[8:], strict=True)
])
display(abertura.round(3))

forma_9z = perfis_major.loc[perfis_major["chave"].eq("9z_globant"), [
    "jogador", "id", "series_equivalentes_adr", "adr_recente",
    "kast_recente", "saldo_mapa_recente", "elo_rival_recente",
]].copy()
assert set(forma_9z["jogador"]) == {"max", "HUASOPEEK", "luchov", "history", "dgt"}
display(forma_9z.round(3))


,jogador,id,identificação,series_equivalentes_adr,adr_recente,kast_recente,saldo_mapa_recente,elo_rival_recente,nivel_eventos_recente,fracao_tier3_recente
20,NAF-FLY,78.0,apelido único no histórico,27.041,76.516,74.750,1.109,1646.897,1.675,0.001
21,EliGE,32.0,apelido único no histórico,27.056,81.577,71.047,0.930,1645.242,1.678,0.000
22,JT,532.0,apelido único no histórico,23.713,64.143,69.197,-2.597,1650.243,1.643,0.003
23,Jorko,25616.0,apelido único no histórico,21.552,62.839,69.530,0.245,1641.354,1.556,0.039
24,malbsMd,881.0,apelido único no histórico,26.581,74.120,71.267,-0.731,1664.618,1.810,0.000


,Equipe A,Equipe B,Vitória de A V3 (%),Vitória de A V4 (%)
0,Ninjas in Pyjamas,M80,59.254,58.280
1,PVISION,DENDELE,56.472,70.439
2,HEROIC,Imperial Esports,52.276,68.172
3,FaZe Clan,Wildcard,52.693,65.738
4,Team Liquid,GOSH,62.803,68.645
5,Eternal Fire,FlyQuest,52.833,54.818
6,100 Thieves,Chinggis Warriors,51.859,61.194
7,9z Globant,Lynn Vision Gaming,52.740,67.798


,jogador,id,series_equivalentes_adr,adr_recente,kast_recente,saldo_mapa_recente,elo_rival_recente
35,max,2846.0,20.255,58.929,68.172,-2.945,1731.020
36,HUASOPEEK,5345.0,20.255,74.371,70.610,0.017,1716.605
37,luchov,3357.0,20.066,73.480,68.854,-0.447,1730.751
38,history,4516.0,0.262,NaN,NaN,NaN,NaN
39,dgt,4393.0,21.529,77.966,73.988,0.778,1726.565


### 8.3 Usar o mesmo motor e a mesma semente para V3 e V4

Cada versão roda **100.000 torneios** com semente **25**. Registramos a campanha de cada time da Etapa 1 dentro de cada simulação V4. A referência HLTV arquivada na tabela tem **20.000 simulações**.

In [22]:
# Rodar V3 e V4 com o mesmo conjunto de regras e registrar os perfis da V4
resultado_ref = json.loads((pasta_major / "ultima_simulacao.json").read_bytes())
assert resultado_ref["modelo"] == selecao_ref["modelo"]
assert resultado_ref["data_previsao"] == configuracao["data_previsao"]
repeticoes = 100_000
semente = 25

print("V3 · 100.000 simulações")
contagens_v3, _, _ = simulador.monte_carlo(
    configuracao, chaves, matrizes_v3, iterations=repeticoes, seed=semente,
)

campanhas = [(3, 0), (3, 1), (3, 2), (2, 3), (1, 3), (0, 3)]
indice_campanha = {campanha: i for i, campanha in enumerate(campanhas)}
equipe_etapa1 = [indice_equipe[chave] for chave in chaves_etapa1]
perfis_etapa1 = []
suico_original = simulador.swiss_stage


def suico_com_registro(equipes, etapa, matrizes, rng, prioridade, registrar):
    if etapa != 1:
        return suico_original(equipes, etapa, matrizes, rng, prioridade, registrar)
    series_jogadas = dict.fromkeys(equipes, 0)

    def registrar_partida(time_a, time_b, etapa_atual):
        series_jogadas[time_a] += 1
        series_jogadas[time_b] += 1
        registrar(time_a, time_b, etapa_atual)

    classificados = suico_original(
        equipes, etapa, matrizes, rng, prioridade, registrar_partida,
    )
    avancaram = set(classificados)
    perfil = []
    for equipe in equipe_etapa1:
        total = series_jogadas[equipe]
        vitorias = 3 if equipe in avancaram else total - 3
        perfil.append(indice_campanha[vitorias, total - vitorias])
    perfis_etapa1.append(tuple(perfil))
    return classificados


print("V4 · 100.000 simulações")
with patch.object(simulador, "swiss_stage", suico_com_registro):
    contagens, _, _ = simulador.monte_carlo(
        configuracao, chaves, matrizes, iterations=repeticoes, seed=semente,
    )
cenarios_etapa1 = np.asarray(perfis_etapa1, dtype=np.int8)
assert cenarios_etapa1.shape == (repeticoes, 16)
for i, vagas in enumerate((2, 3, 3, 3, 3, 2)):
    assert np.all((cenarios_etapa1 == i).sum(axis=1) == vagas)
np.testing.assert_array_equal(
    (cenarios_etapa1 < 3).sum(axis=0), contagens[equipe_etapa1, 0],
)

fases = [
    "avançar à Etapa 2", "avançar à Etapa 3", "quartas",
    "semifinal", "final", "título", "terceiro lugar",
]
chances = pd.DataFrame(100 * contagens / repeticoes, index=chaves, columns=fases)
chances = chances.rename_axis("chave").reset_index().merge(
    cobertura_major[["chave", "nome_oficial", "etapa_entrada"]],
    on="chave", validate="one_to_one",
)
comparacao_v3 = pd.DataFrame({
    "chave": chaves, "V3 (%)": 100 * contagens_v3[:, 0] / repeticoes,
})
pasta_simulacao_ref = pasta_major / PureWindowsPath(resultado_ref["pasta_simulacao"]).name
chances_ref = pd.read_csv(
    pasta_simulacao_ref / "chances_por_equipe.csv", usecols=["chave", "alcancar_etapa2_ou_superior"],
).rename(columns={"alcancar_etapa2_ou_superior": "Referência HLTV (%)"})
chances_ref["Referência HLTV (%)"] *= 100
primeira_fase = (
    chances.loc[chances["etapa_entrada"].eq(1)]
    .merge(comparacao_v3, on="chave", validate="one_to_one")
    .merge(chances_ref, on="chave", validate="one_to_one")
    .rename(columns={"avançar à Etapa 2": "V4 (%)"})
    .sort_values("V4 (%)", ascending=False)
    .reset_index(drop=True)
)
assert len(primeira_fase) == 16
assert np.isclose(primeira_fase[["V3 (%)", "V4 (%)"]].sum().to_numpy(), 800.0).all()
primeira_fase["Diferença (p.p.)"] = primeira_fase["V4 (%)"] - primeira_fase["V3 (%)"]
display(primeira_fase[[
    "nome_oficial", "V4 (%)", "V3 (%)", "Diferença (p.p.)", "Referência HLTV (%)",
]].round(3))

V3 · frequências de 100.000 simulações arquivadas
V4 · 100.000 simulações
Simulacoes: 5000/100000
Simulacoes: 10000/100000
Simulacoes: 15000/100000
Simulacoes: 20000/100000
Simulacoes: 25000/100000
Simulacoes: 30000/100000
Simulacoes: 35000/100000
Simulacoes: 40000/100000
Simulacoes: 45000/100000
Simulacoes: 50000/100000
Simulacoes: 55000/100000
Simulacoes: 60000/100000
Simulacoes: 65000/100000
Simulacoes: 70000/100000
Simulacoes: 75000/100000
Simulacoes: 80000/100000
Simulacoes: 85000/100000
Simulacoes: 90000/100000
Simulacoes: 95000/100000
Simulacoes: 100000/100000


,nome_oficial,V4 (%),V3 (%),Diferença (p.p.),Referência HLTV (%)
0,PVISION,86.229,68.298,17.931,59.200
1,9z Globant,76.849,43.956,32.893,42.200
2,FaZe Clan,74.716,62.321,12.395,65.910
3,Team Liquid,69.926,62.095,7.831,62.945
4,HEROIC,68.996,60.683,8.313,60.960
5,Ninjas in Pyjamas,57.756,57.474,0.282,49.185
6,Eternal Fire,50.519,55.347,-4.828,46.235
7,M80,43.432,42.213,1.219,41.545
8,Lynn Vision Gaming,42.293,45.711,-3.418,45.785
9,100 Thieves,40.611,36.661,3.950,49.610


## 9. Ler a previsão e as combinações

As frequências são estimativas da simulação V4 para a data de corte. Os mesmos cenários alimentam o mapa individual, os mapas por dupla e a combinação completa. As duplas e os perfis completos são contados em cada simulação, com o chaveamento e todos os resultados daquele cenário.

In [23]:
# Resumir a comparação histórica e a mudança na previsão da Liquid
geral = metricas_teste.loc["Todas as séries"]
liquid = primeira_fase.loc[primeira_fase["chave"].eq("liquid")].iloc[0]
print(f"Log Loss no teste · V3 {geral.loc['V3', 'Log Loss']:.4f} · V4 {geral.loc['V4', 'Log Loss']:.4f}")
print(f"Acurácia no teste · V3 {geral.loc['V3', 'Acurácia']:.2%} · V4 {geral.loc['V4', 'Acurácia']:.2%}")
print(f"Liquid · avanço na V3 {liquid['V3 (%)']:.3f}% · avanço na V4 {liquid['V4 (%)']:.3f}%")
print("Oito maiores chances da V4 · " + ", ".join(primeira_fase["nome_oficial"].head(8)))

Log Loss no teste · V3 0.6557 · V4 0.6570
Acurácia no teste · V3 60.54% · V4 60.21%
Liquid · avanço na V3 62.095% · avanço na V4 69.926%
Oito maiores chances da V4 · PVISION, 9z Globant, FaZe Clan, Team Liquid, HEROIC, Ninjas in Pyjamas, Eternal Fire, M80


In [24]:
# Gráfico 1 — chances individuais por campanha na Etapa 1
from html import escape

from IPython.display import HTML

ordem_pelo_resumo = {chave: i for i, chave in enumerate(chaves_etapa1)}
ordem = primeira_fase["chave"].map(ordem_pelo_resumo).to_numpy()
cenarios = cenarios_etapa1[:, ordem]
nomes = primeira_fase["nome_oficial"].tolist()
placares = [f"{vitorias}–{derrotas}" for vitorias, derrotas in campanhas]
contagem_campanhas = np.column_stack(
    [(cenarios == codigo).sum(axis=0) for codigo in range(len(campanhas))]
)
avancos = contagem_campanhas[:, :3].sum(axis=1)
np.testing.assert_array_equal(contagem_campanhas.sum(axis=1), np.full(len(nomes), repeticoes))
np.testing.assert_array_equal(100 * avancos / repeticoes, primeira_fase["V4 (%)"].to_numpy())


def inteiro_pt(numero):
    return f"{numero:,}".replace(",", ".")


def percentual(ocorrencias):
    return f"{100 * ocorrencias / repeticoes:.3f}%".replace(".", ",")


amostra = f"V4 · {inteiro_pt(repeticoes)} simulações · semente {semente}"
estilo_painel = "background:#fff;color:#172a34;padding:8px 4px;font:14px system-ui,sans-serif"
estilo_tabela = (
    "background:#fff;color:#172a34;border-collapse:separate;border-spacing:3px;"
    "font:13px system-ui,sans-serif;font-variant-numeric:tabular-nums;text-align:center"
)
verde = "99,188,151"
roxo = "170,147,215"

cabecalho = "".join(
    f'<th scope="col" style="padding:6px 10px">{placar}</th>' for placar in placares
)
linhas = []
maximos = contagem_campanhas.max(axis=0)
teto = maximos.max()
destaques = []
for coluna in (0, 5):
    lideres = ", ".join(
        escape(nome) for nome, vezes in zip(nomes, contagem_campanhas[:, coluna], strict=True)
        if vezes == maximos[coluna]
    )
    destaques.append(f'<strong>{placares[coluna]}</strong> · {lideres} ({percentual(maximos[coluna])})')

for nome, valores, avanco in zip(nomes, contagem_campanhas, avancos, strict=True):
    celulas = []
    for coluna, ocorrencias in enumerate(valores):
        rgb = verde if coluna < 3 else roxo
        intensidade = 0.12 + 0.88 * ocorrencias / teto
        estilo = f"background:rgba({rgb},{intensidade:.3f});padding:5px 9px;border-radius:4px"
        if coluna in (0, 5) and ocorrencias == maximos[coluna]:
            estilo += ";font-weight:750;outline:2px solid #374c58;outline-offset:-2px"
        celulas.append(f'<td style="{estilo}">{percentual(ocorrencias)}</td>')
    celulas.append(
        f'<td style="background:#e0f2ea;padding:5px 9px;font-weight:700">{percentual(avanco)}</td>'
    )
    linhas.append(
        '<tr><th scope="row" style="text-align:left;padding:5px 10px;white-space:nowrap">'
        f'{escape(nome)}</th>{"".join(celulas)}</tr>'
    )

display(HTML(f"""
<div style="{estilo_painel}">
<h3 style="margin:0 0 4px">1. Chances por time</h3>
<p style="margin:0 0 10px">{amostra}</p>
<p style="margin:0 0 8px">{" &nbsp;|&nbsp; ".join(destaques)}</p>
<div style="overflow-x:auto">
<table style="{estilo_tabela};min-width:720px">
    <thead>
        <tr>
            <th rowspan="2" scope="col" style="text-align:left;padding:6px 10px">Equipe</th>
            <th colspan="3" scope="colgroup" style="background:#dcefe7;padding:6px 10px">Classificação</th>
            <th colspan="3" scope="colgroup" style="background:#ece4f5;padding:6px 10px">Eliminação</th>
            <th rowspan="2" scope="col" style="background:#dcefe7;padding:6px 10px">Avança (V4)</th>
        </tr>
        <tr>{cabecalho}</tr>
    </thead>
    <tbody>{"".join(linhas)}</tbody>
</table>
</div>
</div>
"""))

In [25]:
# Gráfico 2 — duplas 3–0 e 0–3 no mesmo cenário
from collections import Counter

duplas_3_0 = [tuple(linha) for linha in np.where(cenarios == 0)[1].reshape(-1, 2)]
duplas_0_3 = [tuple(linha) for linha in np.where(cenarios == 5)[1].reshape(-1, 2)]
contagem_3_0 = Counter(duplas_3_0)
contagem_0_3 = Counter(duplas_0_3)
contagem_extremos = Counter(zip(duplas_3_0, duplas_0_3, strict=True))
assert all(sum(totais.values()) == repeticoes for totais in (contagem_3_0, contagem_0_3, contagem_extremos))


def nome_da_dupla(dupla):
    return " + ".join(nomes[i] for i in dupla)


def mapa_de_duplas(contagem, placar, cor):
    maior = max(contagem.values())
    lideres = [nome_da_dupla(dupla) for dupla, vezes in contagem.items() if vezes == maior]
    lideres_html = " / ".join(escape(nome) for nome in lideres)
    titulo = "Dupla mais frequente" if len(lideres) == 1 else "Duplas empatadas na maior frequência"
    cabecalho = "".join(
        f'<th scope="col" style="min-width:58px;max-width:80px;padding:6px 3px;'
        f'font-size:11px;line-height:1.2">{escape(nome)}</th>'
        for nome in nomes
    )
    linhas = []
    for i, nome in enumerate(nomes):
        linha = [
            f'<th scope="row" style="text-align:left;white-space:nowrap;padding:4px 8px">'
            f'{escape(nome)}</th>'
        ]
        for j in range(len(nomes)):
            if j <= i:
                linha.append('<td style="background:#f5f7f7"></td>')
                continue
            dupla = (i, j)
            vezes = contagem[dupla]
            intensidade = 0.08 + 0.82 * vezes / maior
            estilo = (
                f"background-color:rgba({cor},{intensidade:.3f});"
                "font-size:11px;padding:5px 3px;white-space:nowrap"
            )
            if vezes == maior:
                estilo += ";font-weight:750;outline:2px solid #374c58;outline-offset:-2px"
            dica = escape(f"{nome_da_dupla(dupla)} · {inteiro_pt(vezes)} simulações")
            linha.append(f'<td style="{estilo}" title="{dica}">{percentual(vezes)}</td>')
        linhas.append(f'<tr>{"".join(linha)}</tr>')

    return f"""
    <section style="margin:14px 0">
        <h4 style="margin:0 0 4px">Ambos {placar}</h4>
        <p style="margin:4px 0 8px">{titulo} · <strong>{lideres_html}</strong>
            · <strong>{percentual(maior)}</strong> ({inteiro_pt(maior)} simulações)</p>
        <div style="overflow-x:auto">
        <table style="{estilo_tabela}" aria-label="Chance dos dois times terminarem {placar}">
            <thead><tr><th scope="col" style="text-align:left;padding:6px 8px">Equipe</th>{cabecalho}</tr></thead>
            <tbody>{"".join(linhas)}</tbody>
        </table>
        </div>
    </section>
    """


linhas_extremos = []
for (invictos, sem_vitorias), vezes in contagem_extremos.most_common(5):
    linhas_extremos.append(
        f'<tr><td style="background:#e5f4ed;padding:6px 10px">{escape(nome_da_dupla(invictos))}</td>'
        f'<td style="background:#eee8f7;padding:6px 10px">{escape(nome_da_dupla(sem_vitorias))}</td>'
        f'<td style="padding:6px 10px">{inteiro_pt(vezes)}</td>'
        f'<td style="font-weight:700;padding:6px 10px">{percentual(vezes)}</td></tr>'
    )

dupla_exemplo = tuple(
    i for i, chave in enumerate(primeira_fase["chave"]) if chave in {"pvision", "liquid"}
)
exemplo_html = ""
if len(dupla_exemplo) == 2:
    vezes_exemplo = contagem_3_0[dupla_exemplo]
    exemplo_html = (
        f'<p><strong>{escape(nome_da_dupla(dupla_exemplo))}</strong> · ambos 3–0 '
        f'em {percentual(vezes_exemplo)} ({inteiro_pt(vezes_exemplo)} simulações)</p>'
    )

display(HTML(f"""
<div style="{estilo_painel}">
    <h3 style="margin:0 0 4px">2. Probabilidades conjuntas por dupla</h3>
    <p style="margin:0 0 10px">{amostra}</p>
    {mapa_de_duplas(contagem_3_0, "3–0", verde)}
    {exemplo_html}
    {mapa_de_duplas(contagem_0_3, "0–3", roxo)}
    <h4 style="margin:18px 0 4px">Duplas 3–0 e 0–3 que mais saíram juntas</h4>
    <div style="overflow-x:auto">
    <table style="{estilo_tabela}">
        <thead><tr><th scope="col">Ambos 3–0</th><th scope="col">Ambos 0–3</th>
        <th scope="col">Simulações</th><th scope="col">Chance conjunta</th></tr></thead>
        <tbody>{"".join(linhas_extremos)}</tbody>
    </table>
    </div>
</div>
"""))

Equipe,PVISION,9z Globant,FaZe Clan,Team Liquid,HEROIC,Ninjas in Pyjamas,Eternal Fire,M80,Lynn Vision Gaming,100 Thieves,DENDELE,FlyQuest,Wildcard,GOSH,Imperial Esports,Chinggis Warriors
PVISION,,"4,708%","6,155%","4,905%","5,468%","4,487%","2,470%","1,587%","0,922%","1,493%","0,000%","0,817%","0,994%","0,603%","0,608%","0,422%"
9z Globant,,,"3,056%","2,687%","2,671%","1,825%","1,464%","0,789%","0,000%","0,881%","0,959%","1,094%","0,794%","0,606%","0,476%","0,691%"
FaZe Clan,,,,"2,556%","3,042%","2,247%","1,550%","0,845%","0,899%","0,829%","0,808%","0,778%","0,000%","0,438%","0,334%","0,431%"
Team Liquid,,,,,"2,675%","1,879%","0,997%","0,820%","0,890%","0,797%","0,757%","0,727%","0,563%","0,000%","0,345%","0,412%"
HEROIC,,,,,,"2,128%","1,174%","0,755%","0,677%","0,726%","0,698%","0,537%","0,452%","0,345%","0,000%","0,299%"
Ninjas in Pyjamas,,,,,,,"0,839%","0,000%","0,000%","0,446%","0,603%","0,307%","0,416%","0,238%","0,272%","0,095%"
Eternal Fire,,,,,,,,"0,382%","0,559%","0,328%","0,358%","0,000%","0,312%","0,208%","0,195%","0,217%"
M80,,,,,,,,,"0,369%","0,273%","0,248%","0,298%","0,241%","0,156%","0,136%","0,156%"
Lynn Vision Gaming,,,,,,,,,,"0,432%","0,318%","0,301%","0,280%","0,211%","0,155%","0,205%"
100 Thieves,,,,,,,,,,,"0,244%","0,351%","0,234%","0,172%","0,124%","0,000%"
